# tag_classifier: NLI scoring on a Colab GPU

Runs step 3 (`classify`) for the two zero-shot NLI models, which are too slow on a laptop CPU
(about 1.7 s per pair; a T4 does 0.03-0.05 s). Everything else runs locally.

**No database access is needed here.** The bundle holds only the package code, the cleaned course
texts, the curricula file, the candidate tags, the review workbooks (for renames) and the score cache;
no `.env`.

Before opening this notebook, on your machine, **after the last change to candidates or renames**:

```
cd pipeline
tag_classifier/.venv/Scripts/python -m tag_classifier.candidates                      # or --family CS-KHMT ...
tag_classifier/.venv/Scripts/python -m tag_classifier.bundle                          # -> tag_classifier/data/colab_bundle.zip
```

Then here: Runtime > Change runtime type > T4 GPU, and run the cells in order. Long runs (many
programs) survive a disconnect: the score cache is saved to Google Drive after every family and model,
and restored when you run the cells again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the bundle (`colab_bundle.zip`)

In [ ]:
from google.colab import files
import os, zipfile

os.chdir("/content")
uploaded = files.upload()  # choose tag_classifier/data/colab_bundle.zip
zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
print(len(os.listdir("/content/tag_classifier/data/candidates")), "candidate files")

## 2. Install (Colab already has torch with CUDA, numpy, openpyxl)

In [ ]:
!pip install -q "transformers>=4.48" "sentence-transformers>=3.3"

## 3. Keep the score cache on Google Drive

The cache is saved to `MyDrive/tag_classifier/scores.sqlite` after every family and model. If a
session stopped, re-running the notebook merges the saved cache back in, so finished pairs are skipped.

In [ ]:
import shutil, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
sys.path.insert(0, "/content")
from tag_classifier.merge_cache import merge

LOCAL = Path("/content/tag_classifier/cache/scores.sqlite")
SAVED = Path("/content/drive/MyDrive/tag_classifier/scores.sqlite")
SAVED.parent.mkdir(parents=True, exist_ok=True)
if SAVED.exists():
    print(merge(LOCAL, SAVED), "scores restored from Drive")


def save_to_drive():
    tmp = SAVED.with_suffix(".tmp")
    shutil.copy(LOCAL, tmp)
    tmp.replace(SAVED)

## 4. Score

Every program family that has a candidate file, both NLI models. Only the courses to tag are scored
(`[scope]` in `config.toml`). Runs in this Python process, so each model loads once.

In [ ]:
import glob
FAMILIES = sorted(os.path.basename(p)[:-5] for p in glob.glob("/content/tag_classifier/data/candidates/*.json"))
MODELS = ["modernbert", "bge-m3-zeroshot"]
BATCH = 64
print(len(FAMILIES), "families:", FAMILIES[:10], "...")

In [ ]:
from tag_classifier.classify import main as classify_main

for model in MODELS:
    for i, family in enumerate(FAMILIES, 1):
        print(f"[{model} {i}/{len(FAMILIES)}]")
        classify_main(["--family", family, "--model", model, "--batch-size", str(BATCH)])
        save_to_drive()

## 5. Download the scores

In [ ]:
!cd /content && zip -q -r colab_scores.zip tag_classifier/cache/scores.sqlite tag_classifier/data/scores
files.download("/content/colab_scores.zip")

## 6. Back on your machine

Unzip `colab_scores.zip` into `pipeline/tag_classifier/data/colab/<date>/` (git-ignored), or copy
`MyDrive/tag_classifier/scores.sqlite` there, then:

```
cd pipeline
tag_classifier/.venv/Scripts/python -m tag_classifier.merge_cache tag_classifier/data/colab/<date>/tag_classifier/cache/scores.sqlite
tag_classifier/.venv/Scripts/python -m tag_classifier.classify --all     # all models; NLI pairs come from the cache
tag_classifier/.venv/Scripts/python -m tag_classifier.evaluate --all
tag_classifier/.venv/Scripts/python -m tag_classifier.review --all
```

`merge_cache` adds the Colab scores without overwriting anything scored locally, and prints how many
pairs of the current candidates are still unscored (non-zero: the bundle was older than the candidates).